# Diagrams and Schematics

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/dtabuena/Resources/blob/main/Art/Diagrams_and_Schematics.ipynb)

Cartoon and schematic recipes for figures and graphical abstracts. Every section runs on random stand-in data, so nothing here depends on a real dataset.

Shared setup runs once below. Each schematic after that is self-contained.

In [ ]:
import warnings
import urllib.request
import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt

response = urllib.request.urlretrieve('https://raw.githubusercontent.com/dtabuena/Resources/main/Matplotlib_Config/drt_mpl_config.py', 'drt_mpl_config.py')
%run drt_mpl_config.py

## 96-Well Plate Cartoon

Left half of each well: ephys. Right half: seq. Pale grey: empty well.

Text-stripped relative to the source figure: tick labels (A-H, 1-12), plate titles, and the summary prints are gone. The legend is kept, since the split-circle glyph is not readable without it.

In [ ]:
"""Cartoon constants. Colors are the binary present/absent pair from the source legend."""

ROW_LABELS = list('ABCDEFGH')
N_ROWS = 8
N_COLS = 12
WELL_RADIUS = 0.45

EMPTY_COLOR = 'whitesmoke'
EPHYS_COLOR = 'k'
EPHYS_ABSENT = 'lightgrey'
SEQ_COLOR = 'k'
SEQ_ABSENT = 'lightgrey'

PLATE_IDS = ['1', '2', '3', '4']
FILL_FRACTION = 0.75
EPHYS_RATE = 0.70
SEQ_RATE = 0.55

RANDOM_SEED = 0

In [ ]:
"""Random stand-in for well_status. Same columns the real build produces:
plate, well, has_ephys, has_seq.
"""

random_generator = np.random.default_rng(RANDOM_SEED)

all_wells = [f'{row_letter}{col_number:02d}'
             for row_letter in ROW_LABELS
             for col_number in range(1, N_COLS + 1)]

well_records = []
for plate_id in PLATE_IDS:
    n_filled = int(round(FILL_FRACTION * len(all_wells)))
    filled_wells = random_generator.choice(all_wells, size=n_filled, replace=False)
    for well_name in filled_wells:
        well_records.append({
            'plate': plate_id,
            'well': well_name,
            'has_ephys': bool(random_generator.random() < EPHYS_RATE),
            'has_seq': bool(random_generator.random() < SEQ_RATE),
        })

well_status = pd.DataFrame.from_records(well_records)
display(well_status.head())

In [ ]:
def well_to_rc(well_name):
    """Convert 'A01' -> (row_index, col_index) zero-based."""
    row_letter = well_name[0]
    if row_letter not in ROW_LABELS:
        raise ValueError(f"Unrecognized row letter '{row_letter}' in well '{well_name}'")
    row_index = ROW_LABELS.index(row_letter)
    col_index = int(well_name[1:]) - 1
    return row_index, col_index


def draw_well(target_axis, col_index, row_index, left_color, right_color):
    """Two Wedge patches make one split circle."""
    target_axis.add_patch(mpl.patches.Wedge((col_index, -row_index), WELL_RADIUS,
                                            90, 270, color=left_color, linewidth=0))
    target_axis.add_patch(mpl.patches.Wedge((col_index, -row_index), WELL_RADIUS,
                                            270, 90, color=right_color, linewidth=0))


def plot_plate(target_axis, plate_id, status_frame):
    # Empty background wells
    for row_index in range(N_ROWS):
        for col_index in range(N_COLS):
            draw_well(target_axis, col_index, row_index, EMPTY_COLOR, EMPTY_COLOR)

    # Occupied wells
    plate_frame = status_frame[status_frame['plate'] == plate_id]
    for _row_label, well_row in plate_frame.iterrows():
        try:
            row_index, col_index = well_to_rc(well_row['well'])
        except (ValueError, IndexError) as parse_error:
            warnings.warn(f"Could not parse well '{well_row['well']}' on plate "
                          f"{plate_id}, skipping: {parse_error}")
            continue
        left_color = EPHYS_COLOR if well_row['has_ephys'] else EPHYS_ABSENT
        right_color = SEQ_COLOR if well_row['has_seq'] else SEQ_ABSENT
        draw_well(target_axis, col_index, row_index, left_color, right_color)

    target_axis.set_xlim(-0.8, N_COLS - 0.2)
    target_axis.set_ylim(-(N_ROWS - 0.2), 0.8)
    target_axis.set_xticks([])
    target_axis.set_yticks([])
    target_axis.set_aspect('equal')

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(2.25, 2))
axes = axes.flatten()

for panel_axis, plate_id in zip(axes, PLATE_IDS):
    plot_plate(panel_axis, plate_id, well_status)

markersize = 4
legend_elements = [
    mpl.patches.Patch(color=EPHYS_COLOR, label='Present'),
    mpl.lines.Line2D([0], [0], marker='o', color='w', fillstyle='left',
                     markerfacecolor=EPHYS_COLOR, markerfacecoloralt=EPHYS_ABSENT,
                     markeredgecolor='none', markersize=markersize, label='Ephys (Left)'),
    mpl.patches.Patch(color=EPHYS_ABSENT, label='Absent'),
    mpl.lines.Line2D([0], [0], marker='o', color='w', fillstyle='right',
                     markerfacecolor=SEQ_COLOR, markerfacecoloralt=SEQ_ABSENT,
                     markeredgecolor='none', markersize=markersize, label='Seq (Right)'),
]
fig.legend(handles=legend_elements, loc='lower center', ncol=2,
           frameon=False, bbox_to_anchor=(0.5, -0.05))

plt.tight_layout()
plt.show()